In [ ]:
DATA_SPLIT_MODE = "summary"  # options: "summary", "extended"


In [ ]:
import os
datasets_dir_path = "/path/to/datasets"
datasets = os.listdir(datasets_dir_path)
datasets

In [ ]:
from collections import Counter
from scipy.stats import chi2_contingency


trainval_datasets = {"DATASET_UUID_REDACTED", "DATASET_UUID_REDACTED"}
test_datasets     = {"DATASET_UUID_REDACTED"}

manu_tv = Counter(); manu_te = Counter()
center_tv = Counter(); center_te = Counter()

In [ ]:
import os
import json
import random
import pydicom
from collections import Counter, defaultdict

# Define the root directory where datasets are stored
datasets_dir_path = "/path/to/datasets"

# Get the list of datasets (patient cases)
datasets = os.listdir(datasets_dir_path)

# Dataset to exclude
excluded_dataset = "DATASET_UUID_REDACTED"

# Dictionaries to store results
patient_data = {}  # Stores per-subject manufacturer and private tag info
overall_private_tag_count = Counter()  # Counts private tags across all datasets
dataset_private_tag_count = {}  # Stores private tag counts per dataset

overall_manufacturer_count = Counter()  # Counts manufacturers across all datasets
dataset_manufacturer_count = {}  # Stores manufacturer counts per dataset

# Iterate over each dataset (patient case)
for dataset in datasets:
    # Skip the excluded dataset
    if dataset == excluded_dataset:
        continue

    dataset_dir_path = os.path.join(datasets_dir_path, dataset)

    # Load the study metadata
    index_file_path = os.path.join(dataset_dir_path, "index.json")
    if not os.path.exists(index_file_path):
        continue

    with open(index_file_path, "r") as f:
        studies = json.load(f)

    # Counters for this dataset
    dataset_private_tag_counter = Counter()
    dataset_manufacturer_counter = Counter()

    # Iterate over studies in the dataset
    for study in studies:
        subject_name = study.get("subjectName", "Unknown")
        series_list = [s for s in study.get("series", []) if s.get("tags") == []]

        if not series_list:
            continue  # No valid series found

        # Randomly select one valid series
        selected_series = random.choice(series_list)
        series_folder = selected_series["folderName"]
        series_path = os.path.join(dataset_dir_path, study["path"], series_folder)

        if not os.path.exists(series_path):
            continue

        # Get list of DICOM files in the selected series
        dicom_files = [f for f in os.listdir(series_path) if f.endswith(".dcm")]
        if not dicom_files:
            continue

        # Randomly select one DICOM file
        dicom_file_path = os.path.join(series_path, random.choice(dicom_files))

        # Read DICOM metadata
        try:
            ds = pydicom.dcmread(dicom_file_path)
            manufacturer = ds.get("Manufacturer", "Unknown")

            # Extract private tag data
            private_tag_value = ds.get((0x70D1, 0x2003), "Not Found")
            if hasattr(private_tag_value, "value"):
                private_tag_value = private_tag_value.value

            # which split
            if dataset in trainval_datasets:
                split = "train_val"
            elif dataset in test_datasets:
                split = "test"
            else:
                split = "unknown"
            
            # manufacturer category (GE / Siemens / Other)
            m = str(manufacturer).upper()
            if "GE" in m:
                m_cat = "GE"
            elif "SIEMENS" in m:
                m_cat = "Siemens"
            else:
                m_cat = "Other"
            
            if split == "train_val":
                manu_tv[m_cat] += 1
            elif split == "test":
                manu_te[m_cat] += 1
            
            # centre category (use your private tag as centre id)
            c = str(private_tag_value).strip()  # e.g. "01" or "03"
            # ?????????????????
            # ???01=La Fe, 02=ULS, 03=CHU Angers?????????????????
            center_map = {"01": "La Fe", "03": "ULS", "06": "CHU Angers"}
            c_cat = center_map.get(c, "Other")
            
            if split == "train_val":
                center_tv[c_cat] += 1
            elif split == "test":
                center_te[c_cat] += 1

            # Store subject-wise manufacturer and private tag info
            patient_data[subject_name] = {
                "Manufacturer": manufacturer,
                "Private Tag": private_tag_value
            }

            # Count occurrences of private tags
            overall_private_tag_count[private_tag_value] += 1
            dataset_private_tag_counter[private_tag_value] += 1

            # Count occurrences of manufacturers
            overall_manufacturer_count[manufacturer] += 1
            dataset_manufacturer_counter[manufacturer] += 1

        except Exception as e:
            print(f"Error reading DICOM file {dicom_file_path}: {e}")

    # Store per-dataset private tag and manufacturer counts
    dataset_private_tag_count[dataset] = dataset_private_tag_counter
    dataset_manufacturer_count[dataset] = dataset_manufacturer_counter

# Print subject-wise manufacturer and private tag info
print("\n=== Subject Manufacturer & Private Tag Information ===")
for subject_name, data in patient_data.items():
    print(f"Subject: {subject_name}, Manufacturer: {data['Manufacturer']}, Private Tag: {data['Private Tag']}")

# Print overall manufacturer statistics
print("\n=== Overall Manufacturer Count Across All Datasets ===")
for manufacturer, count in overall_manufacturer_count.items():
    print(f"Manufacturer '{manufacturer}': {count}")

# Print per-dataset manufacturer statistics
print("\n=== Per-Dataset Manufacturer Count ===")
for dataset, manufacturer_counter in dataset_manufacturer_count.items():
    print(f"\nDataset: {dataset}")
    for manufacturer, count in manufacturer_counter.items():
        print(f"  Manufacturer '{manufacturer}': {count}")

# Print overall private tag statistics
print("\n=== Overall Private Tag Count Across All Datasets ===")
for tag, count in overall_private_tag_count.items():
    print(f"Private Tag '{tag}': {count}")

# Print per-dataset private tag statistics
print("\n=== Per-Dataset Private Tag Count ===")
for dataset, private_tag_counter in dataset_private_tag_count.items():
    print(f"\nDataset: {dataset}")
    for tag, count in private_tag_counter.items():
        print(f"  Private Tag '{tag}': {count}")


In [ ]:
import os
import json
import random
import pydicom

# Define the root directory where datasets are stored
datasets_dir_path = "/path/to/datasets"

# Get the list of datasets (patient cases)
datasets = os.listdir(datasets_dir_path)

#exclude dataset
excluded_dataset = "DATASET_UUID_REDACTED"

# Dictionary to store results
patient_manufacturers = {}

# Iterate over each dataset (patient case)
for dataset in datasets:
    #skip the excluded dataset
    if dataset == excluded_dataset:
        continue
        
    dataset_dir_path = os.path.join(datasets_dir_path, dataset)

    # Load the study metadata
    index_file_path = os.path.join(dataset_dir_path, "index.json")
    if not os.path.exists(index_file_path):
        continue

    with open(index_file_path, "r") as f:
        studies = json.load(f)

    # Iterate over studies in the dataset
    for study in studies:
        study_id = study.get("subjectName", "Unknown")
        series_list = [s for s in study.get("series", []) if s.get("tags") == []]

        if not series_list:
            continue  # No valid series found

        # Randomly select one valid series
        selected_series = random.choice(series_list)
        series_folder = selected_series["folderName"]
        series_path = os.path.join(dataset_dir_path, study["path"], series_folder)

        if not os.path.exists(series_path):
            continue

        # Get list of DICOM files in the selected series
        dicom_files = [f for f in os.listdir(series_path) if f.endswith(".dcm")]
        if not dicom_files:
            continue

        # Randomly select one DICOM file
        dicom_file_path = os.path.join(series_path, random.choice(dicom_files))

        # Read DICOM metadata
        try:
            ds = pydicom.dcmread(dicom_file_path)
            manufacturer = ds.get("Manufacturer", "Unknown")

            # Extract private tag data
            private_tag_value = ds.get((0x70D1, 0x2003), "Not Found")
            if hasattr(private_tag_value, "value"):
                private_tag_value = private_tag_value.value

            # Store in dictionary
            patient_manufacturers[study_id] = {
                "Manufacturer": manufacturer,
                "Private Tag": private_tag_value
            }

        except Exception as e:
            print(f"Error reading DICOM file {dicom_file_path}: {e}")

# Print the results
for study_id, data in patient_manufacturers.items():
    print(f"Study ID: {study_id}, Manufacturer: {data['Manufacturer']}, Private Tag: {data['Private Tag']}")


In [ ]:
from scipy.stats import chi2_contingency

# manufacturer p
cats_m = sorted(set(manu_tv) | set(manu_te))
table_m = [[manu_tv[c] for c in cats_m],
           [manu_te[c] for c in cats_m]]
chi2, p_m, dof, exp = chi2_contingency(table_m)
print("Manufacturer table:", cats_m, table_m, "p =", p_m)

# centre p
cats_c = sorted(set(center_tv) | set(center_te))
table_c = [[center_tv[c] for c in cats_c],
           [center_te[c] for c in cats_c]]
chi2, p_c, dof, exp = chi2_contingency(table_c)
print("Centre table:", cats_c, table_c, "p =", p_c)


In [ ]:
for i in range(0, len(datasets)):
    dataset_dir_path = os.path.join(datasets_dir_path, dataset)
    

In [ ]:
dataset = datasets[2]
dataset_dir_path = os.path.join(datasets_dir_path, dataset)
print(dataset_dir_path)
contents = os.listdir(dataset_dir_path)
print(contents[0:5])

In [ ]:
import json
index_file_path = os.path.join(dataset_dir_path, "index.json")
with open(index_file_path) as f:
    studies = json.load(f)

In [ ]:
studies[2]

In [ ]:
studies[2]['series'][0]

In [ ]:
studies[2]['series'][0]['tags']

In [ ]:
series = studies[2]['series'][0]
serie_dir_path = os.path.join(dataset_dir_path, studies[2]["path"], series["folderName"])
images = [file for file in os.listdir(serie_dir_path) if file.endswith(".dcm")]
print(f"{len(images)} images in that series")
print(images[0:3])

In [ ]:
import matplotlib.pyplot as plt
import pydicom
dcm_file_path = os.path.join(serie_dir_path, images[3])
ds = pydicom.dcmread(dcm_file_path)
# plt.imshow(ds.pixel_array, cmap=plt.cm.bone)

In [ ]:
ds

In [ ]:
manufacturer = ds.get("Manufacturer","Unknown")


In [ ]:
ds.get("Manufacturer","Unknown")

In [ ]:
ds.get("Manufacturer")

In [ ]:
ds.get((0x70D1, 0x2003),"tag not found")